In [1]:
from pathlib import Path
import json
import hashlib
from datetime import datetime
import pandas as pd

In [2]:
project_root = Path.cwd().parent

metadata_dir = (
    project_root
    / "data"
    / "metadata"
)

processed_v1_dir = (
    project_root
    / "data"
    / "processed"
    / "v1"
)

processed_v1_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
RANDOM_SEED = 42
PREPROCESSING_VERSION = "v1"
DATASET_VERSION = "v1"

In [4]:
split_files = [
    "svm_closed_set_enrollment.csv",
    "svm_closed_set_train.csv",
    "svm_closed_set_validation.csv",
    "svm_closed_set_test.csv",
    "cosine_validation_enrollment.csv",
    "cosine_validation_query.csv",
    "cosine_validation_unknown.csv",
    "cosine_test_enrollment.csv",
    "cosine_test_query.csv",
    "cosine_test_unknown.csv"
]

In [5]:
def calculate_file_sha256(file_path):
    sha256 = hashlib.sha256()
    with open(
        file_path,
        "rb"
    ) as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)
    return sha256.hexdigest()

In [6]:
import shutil

v1_metadata_dir = (
    processed_v1_dir
    / "metadata"
)

v1_metadata_dir.mkdir(
    parents=True,
    exist_ok=True
)

for file in split_files:
    src = (
        metadata_dir
        / file
    )
    dst = (
        v1_metadata_dir
        / file
    )
    shutil.copy2(
        src,
        dst
    )

In [8]:
split_summary = {}

for file in split_files:

    path = (
        v1_metadata_dir
        / file
    )

    df = pd.read_csv(path)

    split_summary[file] = {

        "num_audio":
            len(df),

        "num_speaker":
            df["speaker_id"].nunique(),

        "checksum":

            calculate_file_sha256(
                path
            )
    }

In [9]:
manifest = {
    "dataset_version":
        DATASET_VERSION,

    "created_at":
        datetime.now().isoformat(),

    "random_seed":
        RANDOM_SEED,

    "preprocessing_version":
        PREPROCESSING_VERSION,

    "freeze_status":
        "FROZEN",

    "rule":
        [
            "Do not change speaker assignment",
            "Do not change split assignment",
            "Create v2 for any modification"
        ],

    "splits":
        split_summary
}

In [11]:
manifest_path = (
    processed_v1_dir
    / "split_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    manifest_path
)

d:\HCMUS\HOCTAP\Semesters\25-26HK3\HocThongKe\Project\VoiceStudy-Assistant\data\processed\v1\split_manifest.json


In [12]:
with open(
    manifest_path,
    "r",
    encoding="utf-8"
) as f:

    manifest_check = json.load(f)

manifest_check

{'dataset_version': 'v1',
 'created_at': '2026-08-04T03:54:15.649512',
 'random_seed': 42,
 'preprocessing_version': 'v1',
 'freeze_status': 'FROZEN',
 'rule': ['Do not change speaker assignment',
  'Do not change split assignment',
  'Create v2 for any modification'],
 'splits': {'svm_closed_set_enrollment.csv': {'num_audio': 50,
   'num_speaker': 10,
   'checksum': '4a48344c43713b15856259b972be5b902c9d3b315611f3e2962d8c1268fee925'},
  'svm_closed_set_train.csv': {'num_audio': 100,
   'num_speaker': 10,
   'checksum': 'eca4a5444cc24e10dc5cf00e70e493b19720b6b28b8489ee30c102dd83029f7f'},
  'svm_closed_set_validation.csv': {'num_audio': 50,
   'num_speaker': 10,
   'checksum': '81c26de9bdb28050c2375ade62fbb18215765a87e92f693e499e7a53b7648594'},
  'svm_closed_set_test.csv': {'num_audio': 50,
   'num_speaker': 10,
   'checksum': '68f8230be2aca57ccc52029d2c66121790536f39ab3a22836b79e4ddffd4f523'},
  'cosine_validation_enrollment.csv': {'num_audio': 10,
   'num_speaker': 2,
   'checksum': 'b

In [13]:
required_files = [
    processed_v1_dir / "split_manifest.json"
]


for f in required_files:

    print(
        f,
        f.exists()
    )

d:\HCMUS\HOCTAP\Semesters\25-26HK3\HocThongKe\Project\VoiceStudy-Assistant\data\processed\v1\split_manifest.json True
